# Fine-tune AlephBERT for your own Hebrew classification task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spivi/alephbert-intent-he/blob/main/finetune.ipynb)

This notebook takes a CSV of labelled Hebrew text and fine-tunes
[AlephBERT](https://huggingface.co/onlplab/alephbert-base) into a classifier for
**your** labels, then exports it to ONNX for CPU inference.

It's the generalized version of the recipe behind
[spivi87/alephbert-intent-he](https://huggingface.co/spivi87/alephbert-intent-he)
(which beats GPT-4o-mini zero-shot 57% → 74% on Hebrew shopping intents, free per call).

**Runtime:** set `Runtime → Change runtime type → T4 GPU`. End-to-end is ~10–25 min
depending on dataset size.

**Your CSV needs two columns:** `text` (Hebrew) and `label` (any strings, they
become the classes).

## 1. Install dependencies

In [ ]:
%pip install -q "transformers>=4.40" "torch>=2.3" "datasets>=2.19" "scikit-learn>=1.4" "onnx>=1.16" "onnxruntime>=1.17" "onnxscript>=0.1"
print("deps installed")

## 2. Configuration

`BASE_MODEL` defaults to `onlplab/alephbert-base`, the right base when you're
training a **new** label set (you get a fresh classification head sized to your
classes).

> If instead you want to *extend the exact shopping-intent label set* of
> `spivi87/alephbert-intent-he`, set `BASE_MODEL` to that repo. For any other
> task, keep the default, a head trained on 17 shopping intents won't transfer
> to your labels.

In [ ]:
BASE_MODEL   = "onlplab/alephbert-base"   # see note above
MAX_LENGTH   = 128
EPOCHS       = 10
BATCH_SIZE   = 16
LEARNING_RATE = 2e-5
TEST_SIZE    = 0.2     # held-out fraction (stratified by label)
SEED         = 42
OUTPUT_DIR   = "my-hebrew-classifier"

## 3. Upload your CSV

Upload a CSV with `text` and `label` columns. If you just want to try the
notebook, run the next cell and choose **Cancel**, it falls back to a tiny
built-in Hebrew sample so the whole flow runs end to end.

In [ ]:
import io
import pandas as pd

df = None
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if uploaded:
        name = next(iter(uploaded))
        df = pd.read_csv(io.BytesIO(uploaded[name]))
        print(f"Loaded {name}: {len(df)} rows")
except Exception as e:
    print(f"(upload skipped: {e})")

if df is None:
    print("No CSV uploaded, using a tiny built-in sample so the notebook runs.")
    sample = [
        ("תוסיף חלב וביצים", "ADD_ITEM"), ("צריך לקנות לחם", "ADD_ITEM"),
        ("תן לי 2 קילו עגבניות", "ADD_ITEM"), ("חלב, גבינה, יוגורט", "ADD_ITEM"),
        ("מה ברשימה?", "SHOW_LIST"), ("תראה לי את הרשימה", "SHOW_LIST"),
        ("מה צריך לקנות", "SHOW_LIST"), ("הצג רשימה", "SHOW_LIST"),
        ("סיימתי קניות", "DONE"), ("קניתי הכל", "DONE"),
        ("גמרתי", "DONE"), ("הכל נקנה", "DONE"),
        ("תודה", "OTHER"), ("שלום", "OTHER"), ("מה השעה", "OTHER"), ("בוקר טוב", "OTHER"),
    ]
    df = pd.DataFrame(sample, columns=["text", "label"])

assert {"text", "label"}.issubset(df.columns), "CSV must have 'text' and 'label' columns"
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
print(df["label"].value_counts())

## 4. Build the label map from your data

Labels are read directly from the CSV, sorted for determinism so the same data
always yields the same `id2label`.

⚠️ **If your CSV was generated by LLM paraphrasing** (like the parent project),
make sure paraphrases of the same source example don't straddle the train/test
split, otherwise you'll leak and over-report accuracy. The stratified split
below splits at the *row* level, which is correct for genuinely independent rows.

In [ ]:
labels = sorted(df["label"].unique())
label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for lab, i in label2id.items()}
num_labels = len(labels)
df["label_id"] = df["label"].map(label2id)
print(f"{num_labels} classes: {labels}")

## 5. Tokenize + stratified train/test split

In [ ]:
import numpy as np
import random
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["label_id"], random_state=SEED
)
print(f"train={len(train_df)}  test={len(test_df)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_ds(frame):
    ds = Dataset.from_dict({"text": frame["text"].tolist(),
                            "labels": frame["label_id"].tolist()})
    return ds.map(
        lambda b: tokenizer(b["text"], max_length=MAX_LENGTH,
                            truncation=True, padding="max_length"),
        batched=True, remove_columns=["text"],
    ).with_format("torch")

train_ds, test_ds = to_ds(train_df), to_ds(test_df)

## 6. Fine-tune (same hyperparameters as the parent project)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=num_labels, id2label=id2label, label2id=label2id,
)

def metrics(eval_pred):
    logits, gold = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(gold, preds),
            "f1_weighted": f1_score(gold, preds, average="weighted", zero_division=0)}

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_accuracy",
    greater_is_better=True, logging_steps=25, save_total_limit=1,
    report_to="none", seed=SEED, data_seed=SEED,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 7. Evaluate: accuracy, F1, and a per-class report

In [ ]:
from sklearn.metrics import classification_report

pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids
print(f"Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Macro F1:    {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}\n")
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

## 8. Export to ONNX (CPU-fast)

`dynamo=False` is **required**: the newer dynamo exporter silently drops weights
for BERT-style models (you get a tiny ONNX file with near-random accuracy). The
legacy exporter is correct.

In [ ]:
import torch

model.eval()
dummy = tokenizer("דוגמה", return_tensors="pt", max_length=MAX_LENGTH,
                  truncation=True, padding="max_length")
onnx_path = f"{OUTPUT_DIR}/model.onnx"
torch.onnx.export(
    model, (dummy["input_ids"], dummy["attention_mask"]), onnx_path,
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"},
                  "attention_mask": {0: "batch", 1: "seq"},
                  "logits": {0: "batch"}},
    opset_version=14, dynamo=False,
)

# Validate ONNX matches PyTorch, and that PAD masking is correct
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
enc = tokenizer(test_df["text"].iloc[0], return_tensors="np", max_length=MAX_LENGTH,
                truncation=True, padding="max_length")
onnx_logits = sess.run(None, {"input_ids": enc["input_ids"].astype("int64"),
                              "attention_mask": enc["attention_mask"].astype("int64")})[0]
print("ONNX OK, predicted:", id2label[int(onnx_logits[0].argmax())])

## 9. Download your model

Zips the model directory (PyTorch weights + tokenizer + `config.json` with your
`id2label` baked in, so `pipeline()` returns named labels + the ONNX file).

In [ ]:
import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
try:
    from google.colab import files  # type: ignore
    files.download(f"{OUTPUT_DIR}.zip")
except Exception:
    print(f"Saved {OUTPUT_DIR}.zip, download it from the file browser.")

## Next steps

- Load it anywhere: `pipeline("text-classification", model="./my-hebrew-classifier")`
- Push to the Hub: `model.push_to_hub("you/your-model")`
- Tune for your task: more classes → more data per class + a higher confidence
  threshold; route low-confidence predictions to an LLM fallback.

Full recipe, baselines, and evaluation tooling:
**[github.com/spivi/alephbert-intent-he](https://github.com/spivi/alephbert-intent-he)**